# MTM1M3 — Hardpoint Breakaway: Stiffness & Residual
**Author**: Noah Gonzalez  
**Date**: 2026-03-17

This notebook contains the scripts required to extract physical properties for a specific date range or a single day.  
  
When running this notebook, the following outputs are generated:
- A CSV file containing the processed data.
- Force measurement (N) vs. Time plots.
- Stiffness distribution plots.
- Residual analysis plots.

In [ ]:
import asyncio
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.dates as mdates

from astropy.time import Time
from scipy.optimize import curve_fit
from scipy.special import erf

from lsst.summit.utils.efdUtils import makeEfdClient
from lsst.ts.xml.enums.MTM1M3 import HardpointTest

In [ ]:
# CONFIGURATION

efd_client = makeEfdClient("usdf_efd")
BASE_TOPIC = "lsst.sal.MTM1M3"
N_HP = 6
M_TO_UM = 1e6

# Force bands
COMP_LIMITS = (2981, 3959)  # N  compression
TENS_LIMITS = (-4420, -3456)  # N  tension
SPEC_STIFFNESS = 100  # N/µm  max limit stiffness for breakaway force

# Plot window
DISPLACEMENT_CROP_RANGE = 500  # µm

# Stiffness fit
DISPLACEMENT_CROP_RANGE_FOR_FIT = 100  # region used for stiffness fit in µm
FIT_POINTS_AROUND_ZERO = 10  # ±N points around zero-displacement
FIT_PAD_S = 0.2  # padding around state segment edges

# Auto-detect windows
TEST_EVENT_TOPIC = f"{BASE_TOPIC}.logevent_hardpointTestStatus"
MIN_TEST_DURATION_S = 60  # seconds
WINDOW_GAP_S = 30  # seconds

# Session splitting
SESSION_GAP_S = 300  # seconds between sessions

STATUS_COLORS = {
    HardpointTest.NOTTESTED: "lightgrey",
    HardpointTest.MOVINGNEGATIVE: "lightsteelblue",
    HardpointTest.TESTINGPOSITIVE: "forestgreen",
    HardpointTest.TESTINGNEGATIVE: "royalblue",
    HardpointTest.MOVINGREFERENCE: "darkseagreen",
    HardpointTest.PASSED: "dimgrey",
    HardpointTest.FAILED: "red",
}

# Pointing telemetry topics
EL_AZ_PRIMARY_EL = "lsst.sal.MTMount.elevation"
EL_AZ_PRIMARY_AZ = "lsst.sal.MTMount.azimuth"
DEFAULT_SALINDEX = "N/A"

In [ ]:
# UTILITIES


def ensure_utc_index(df: pd.DataFrame) -> pd.DataFrame:
    # Normalise any DatetimeIndex to UTC and sort by time.
    if df is None or df.empty:
        return df
    if not isinstance(df.index, pd.DatetimeIndex):
        df.index = pd.to_datetime(df.index)
    if df.index.tz is None:
        df.index = df.index.tz_localize("UTC")
    else:
        df.index = df.index.tz_convert("UTC")
    return df.sort_index()


def save_fig(fig, outdir, fname, dpi=180):
    """Save figure to *outdir/fname*. If *outdir* is None, do nothing."""
    if outdir is None:
        return
    os.makedirs(outdir, exist_ok=True)
    path = os.path.join(outdir, fname)
    fig.savefig(path, dpi=dpi, bbox_inches="tight")
    print("Saved:", path)


def save_dataframe(df: pd.DataFrame, path: str, fmt: str = "csv") -> None:
    """Save *df* to disk in CSV or Parquet format.

    Args:
        df   : DataFrame.
        path : Full output path.
        fmt  : 'csv' (default) or 'parquet'.
    """
    if df is None or df.empty:
        print("[save_dataframe] Nothing to save — DataFrame is empty.")
        return
    # Ensure parent directory exists even for relative paths like "results/foo.csv"
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    if fmt == "parquet":
        # Normalise extension so the file is always openable without guessing
        if not path.endswith(".parquet"):
            path = os.path.splitext(path)[0] + ".parquet"
        df.to_parquet(path, index=False)
    else:
        if not path.endswith(".csv"):
            path = os.path.splitext(path)[0] + ".csv"
        df.to_csv(path, index=False)
    print(f"DataFrame saved → {path}  ({len(df)} rows × {len(df.columns)} cols)")

In [ ]:
# STATE SEGMENTATION


def build_state_segments(status_series: pd.Series, t0: pd.Timestamp, t1: pd.Timestamp):
    """Return [(start, end, state), ...] for state changes within [t0, t1]."""
    if status_series is None or status_series.empty:
        return []
    s = status_series.dropna().astype(int)
    s = ensure_utc_index(s.to_frame("st"))["st"]
    s = s[(s.index >= t0) & (s.index <= t1)]
    if s.empty:
        return []

    if s.index[0] > t0:
        s.loc[t0] = s.iloc[0]
    if s.index[-1] < t1:
        s.loc[t1] = s.iloc[-1]
    s = s.sort_index()
    # Mark every row where the integer state changes; np.where gives us
    # the positional indices of those transitions.
    change = s.ne(s.shift())
    idx = np.where(change.to_numpy())[0]
    segs = []
    for i in range(len(idx)):
        a = s.index[idx[i]]
        # Next transition start is this segment's end; last segment ends at t1.
        b = s.index[idx[i + 1]] if (i + 1) < len(idx) else t1
        segs.append((a, b, int(s.iloc[idx[i]])))
    return segs


def pick_segment_for_state(segs, target_state: int):
    # if multiple segments have the same state, pick
    # the longest one that has a valid duration (b > a).
    candidates = [(a, b) for (a, b, st) in segs if st == target_state and b > a]
    return max(candidates, key=lambda ab: ab[1] - ab[0]) if candidates else None


def slice_actuator_by_segment(act_df: pd.DataFrame, seg, pad_s: float = FIT_PAD_S):
    """Slice F-D data to segment bounds (pad excludes transition transients)."""
    if seg is None or act_df is None or act_df.empty:
        return pd.DataFrame()
    a, b = seg
    # Shrink the window slightly inward to avoid the ramp transients that
    # occur when the controller switches state (typically < 0.2 s).
    a = a - pd.Timedelta(seconds=pad_s)
    b = b + pd.Timedelta(seconds=pad_s)
    return act_df[(act_df.index >= a) & (act_df.index <= b)].copy()

In [ ]:
# STIFFNESS FIT


def center_df_force_disp(df: pd.DataFrame) -> pd.DataFrame:
    """Shift (displacement, force) so the point of minimum |force| becomes (0, 0)."""
    df = df.sort_index().copy()
    # The quasi-static equilibrium point is where |F| is smallest; removing it
    # lets us fit stiffness through the origin without a free intercept term.
    i0 = df["force"].abs().idxmin()
    x0 = float(df.loc[i0, "displacement"])
    y0 = float(df.loc[i0, "force"])
    df["displacement"] -= x0
    df["force"] -= y0
    return df


def slope_through_origin(x: np.ndarray, y: np.ndarray) -> float:
    """OLS slope for y = k*x (no intercept). Returns NaN if degenerate."""
    # Closed-form OLS for the no-intercept model: k = (x·y) / (x·x)
    denom = float(np.dot(x, x))
    if denom == 0 or not np.isfinite(denom):
        return np.nan
    return float(np.dot(x, y) / denom)


def fit_stiffness_centered(df_state: pd.DataFrame):
    """Estimate stiffness (N/µm) from a centred force-displacement segment.

    Returns:
        k        : float           stiffness in N/µm  (NaN if fit failed)
        df_plot  : pd.DataFrame    centred & cropped data for plotting
        info     : dict            slope, intercept, n_window, n_inliers, sigma
    """
    empty = (np.nan, pd.DataFrame(), {})
    if df_state is None or df_state.empty or len(df_state) < 20:
        return empty

    dfc = center_df_force_disp(df_state)

    # Wide crop for the plot; narrow crop for the linear fit region.
    df_plot = dfc[dfc["displacement"].abs() <= DISPLACEMENT_CROP_RANGE].copy()
    if df_plot.empty or len(df_plot) < 10:
        return np.nan, df_plot, {}

    df_center = df_plot[
        df_plot["displacement"].abs() <= DISPLACEMENT_CROP_RANGE_FOR_FIT
    ].copy()
    if df_center.empty or len(df_center) < 5:
        # Fall back to the full plot window if the narrow region is too sparse
        df_center = df_plot

    x = df_center["displacement"].to_numpy(float)
    # Take a symmetric ±FIT_POINTS_AROUND_ZERO window centred on the
    # row closest to zero displacement; this focuses the fit on the
    # most linear part of the F-D curve and avoids non-linear tails.
    j0 = int(np.argmin(np.abs(x)))
    lo = max(0, j0 - FIT_POINTS_AROUND_ZERO)
    hi = min(len(df_center), j0 + FIT_POINTS_AROUND_ZERO + 1)
    df_fit = df_center.iloc[lo:hi]

    if len(df_fit) < 5:
        return np.nan, df_plot, {}

    xf = df_fit["displacement"].to_numpy(float)
    yf = df_fit["force"].to_numpy(float)
    k = slope_through_origin(xf, yf)

    info = dict(
        slope=k,
        intercept=0.0,
        n_window=len(df_fit),
        n_inliers=len(df_fit),
        sigma=np.nan,
    )
    return k, df_plot, info

In [ ]:
# ERF RESIDUAL MODEL


def erf_model(x, a, x0, w, b, c):
    # Sigmoid-like model: amplitude *a*, centre *x0*, width *w*,
    # vertical offset *b*, and a linear tilt *c* to account for
    # the underlying stiffness slope in the raw (non-centred) data.
    return a * erf((x - x0) / w) + b + c * x


def fit_erf_curve(x, y):
    """Fit erf_model with data-driven bounds and initial guesses."""
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    ymin, ymax = np.nanmin(y), np.nanmax(y)
    xmin, xmax = np.nanmin(x), np.nanmax(x)
    x_span = max(xmax - xmin, 1.0)
    y_span = max(ymax - ymin, 1.0)
    # Initial guesses derived from data statistics to avoid scale mismatch
    a0 = 0.5 * y_span
    b0 = 0.5 * (ymax + ymin)
    idx0 = int(np.argmin(np.abs(y - b0)))  # row closest to the midpoint
    x0_0 = float(x[idx0])
    w0 = max(x_span * 0.15, 1.0)  # erf width ≈ 15 % of x range
    c0 = 0.0
    # Bounds are ×10 wider than the data scale to give the solver room
    # without letting it wander into physically absurd territories.
    a_mag = max(abs(a0) * 10, 1.0)
    b_mag = max(abs(b0) * 10 + y_span, 1.0)
    x0_mag = max(abs(x0_0) + x_span, 1.0)
    w_mag = max(x_span * 5, 10.0)
    c_mag = max(y_span / x_span * 10, 1.0)
    lower = [-a_mag, -x0_mag, 1.0, -b_mag, -c_mag]
    upper = [a_mag, x0_mag, w_mag, b_mag, c_mag]
    # Clip p0 strictly inside bounds; curve_fit rejects p0 on the boundary
    p0 = [
        float(np.clip(v, lower[i] + 1e-6, upper[i] - 1e-6))
        for i, v in enumerate([a0, x0_0, w0, b0, c0])
    ]
    popt, _ = curve_fit(erf_model, x, y, p0=p0, bounds=(lower, upper), maxfev=20_000)
    return popt


def plot_error_residual(ax_fd, ax_res, df_pos, df_neg, title, max_markers=30):
    def plot_one(df, label, color):
        df = df.dropna().sort_index()
        x = df["displacement"].to_numpy(float)
        y = df["force"].to_numpy(float)
        if len(df) < 20:
            # Not enough points for a reliable erf fit; plot raw data only
            ax_fd.plot(x, y, "-", color=color, linewidth=2, label=label)
            return
        # Thin markers so the line stays readable at high sample rates
        step = max(1, len(df) // max_markers)
        popt = fit_erf_curve(x, y)
        y_fit = erf_model(x, *popt)
        resid = y - y_fit
        ax_fd.plot(
            x,
            y,
            "-",
            color=color,
            alpha=0.75,
            linewidth=2,
            marker="o",
            markersize=3,
            markevery=step,
            markeredgewidth=0,
            label=label,
        )
        ax_fd.plot(x, y_fit, "--", color=color, linewidth=2.5, label=f"{label} fit")
        ax_res.plot(
            x, resid, "-", color=color, linewidth=2, label=f"Residual - {label}"
        )

    ax_fd.axhspan(COMP_LIMITS[0], COMP_LIMITS[1], color="lightgrey", alpha=0.30)
    ax_fd.axhspan(TENS_LIMITS[0], TENS_LIMITS[1], color="yellow", alpha=0.15)
    ax_fd.axvline(0, color="black", linewidth=1, alpha=0.6)
    ax_fd.axhline(0, color="black", linewidth=1, alpha=0.6)
    ax_fd.text(
        -DISPLACEMENT_CROP_RANGE * 0.95,
        np.mean(COMP_LIMITS),
        "Compression",
        ha="left",
        va="center",
        fontsize=9,
        fontweight="bold",
    )
    ax_fd.text(
        -DISPLACEMENT_CROP_RANGE * 0.95,
        np.mean(TENS_LIMITS),
        "Tension",
        ha="left",
        va="center",
        fontsize=9,
        fontweight="bold",
    )
    ax_res.axvline(0, color="black", linewidth=1, alpha=0.6)
    ax_res.axhline(0, color="black", linewidth=1, alpha=0.6)

    plot_one(df_pos, "Testing Positive", STATUS_COLORS[HardpointTest.TESTINGPOSITIVE])
    plot_one(df_neg, "Testing Negative", STATUS_COLORS[HardpointTest.TESTINGNEGATIVE])

    for ax, ylabel in [(ax_fd, "Force (N)"), (ax_res, "Residual (data − fit)")]:
        ax.set_title(title, fontsize=9)
        ax.set_xlabel("Displacement (µm)", fontsize=9)
        ax.set_ylabel(ylabel, fontsize=9)
        ax.grid(True, alpha=0.25, linestyle="--")
        ax.legend(
            loc="upper left" if ax is ax_fd else "lower right", fontsize=8, frameon=True
        )

In [ ]:
# EFD QUERIES


async def fetch_status(t0: pd.Timestamp, t1: pd.Timestamp) -> pd.DataFrame:
    # Pull all six HP state columns in a single EFD query instead of one per HP.
    df = await efd_client.select_time_series(
        topic_name=f"{BASE_TOPIC}.logevent_hardpointTestStatus",
        fields=[f"testState{i}" for i in range(N_HP)],
        start=Time(t0.to_pydatetime(), scale="utc"),
        end=Time(t1.to_pydatetime(), scale="utc"),
    )
    if df is None or df.empty:
        return pd.DataFrame()
    return ensure_utc_index(df)


async def fetch_actuator_force_disp(
    t0: pd.Timestamp, t1: pd.Timestamp, hp_idx: int
) -> pd.DataFrame:
    df = await efd_client.select_time_series(
        topic_name=f"{BASE_TOPIC}.hardpointActuatorData",
        fields=[f"measuredForce{hp_idx}", f"displacement{hp_idx}"],
        start=Time(t0.to_pydatetime(), scale="utc"),
        end=Time(t1.to_pydatetime(), scale="utc"),
    )
    if df is None or df.empty:
        return pd.DataFrame()
    df = ensure_utc_index(df).rename(
        columns={
            f"measuredForce{hp_idx}": "force",
            f"displacement{hp_idx}": "disp_m",
        }
    )
    # Convert displacement from metres to micrometres for plotting
    df["displacement"] = df["disp_m"] * M_TO_UM
    return df[["force", "displacement"]].dropna().sort_index()


async def fetch_all_actuators(
    t0: pd.Timestamp, t1: pd.Timestamp
) -> dict[int, pd.DataFrame]:
    """Fetch force/displacement for all HPs concurrently.

    Returns {hp_idx (0-based): DataFrame}.
    asyncio.gather fires all N_HP coroutines simultaneously, reducing
    total wall-clock time from N×RTT to roughly 1×RTT.
    """
    tasks = [fetch_actuator_force_disp(t0, t1, hp_idx) for hp_idx in range(N_HP)]
    results = await asyncio.gather(*tasks, return_exceptions=True)
    out = {}
    for hp_idx, res in enumerate(results):
        if isinstance(res, Exception):
            # Log and continue; callers check for empty DataFrames
            print(f"  [fetch] HP{hp_idx + 1} actuator error: {res}")
            out[hp_idx] = pd.DataFrame()
        else:
            out[hp_idx] = res
    return out

In [ ]:
# ANGLE QUERIES


async def fetch_elevation_azimuth(
    t0: pd.Timestamp, t1: pd.Timestamp
) -> tuple[float, float]:
    """Return median (elevation_deg, azimuth_deg) over [t0, t1].

    Queries MTMount actual encoder positions (actualPosition).
    Returns (nan, nan) if no data is available.
    """
    t0_ast = Time(t0.to_pydatetime(), scale="utc")
    t1_ast = Time(t1.to_pydatetime(), scale="utc")

    try:
        # Query both axes concurrently — they are independent topics
        df_el, df_az = await asyncio.gather(
            efd_client.select_time_series(
                EL_AZ_PRIMARY_EL, fields=["actualPosition"], start=t0_ast, end=t1_ast
            ),
            efd_client.select_time_series(
                EL_AZ_PRIMARY_AZ, fields=["actualPosition"], start=t0_ast, end=t1_ast
            ),
        )
        if (
            df_el is not None
            and not df_el.empty
            and df_az is not None
            and not df_az.empty
        ):
            # Median is robust to occasional encoder glitches
            return (
                float(np.nanmedian(df_el["actualPosition"].to_numpy(float))),
                float(np.nanmedian(df_az["actualPosition"].to_numpy(float))),
            )
    except Exception as exc:
        print(f"  [el/az] MTMount query failed ({exc}).")

    print(f"  [el/az] No pointing data for window {t0} – {t1}.")
    return float("nan"), float("nan")


def fetch_salindex_for_window(*args, **kwargs) -> str:
    """SALindex is not reliably available in the EFD.
    Pass salindex='<value>' manually to run_day_plots() when needed.
    """
    return DEFAULT_SALINDEX


In [ ]:
# AUTO-DETECT TEST WINDOWS FOR A DATE RANGE


async def detect_test_windows(
    date_start: str,
    date_end: str,
    min_duration_s: float = MIN_TEST_DURATION_S,
    gap_s: float = WINDOW_GAP_S,
    verbose: bool = True,
) -> pd.DataFrame:
    """Detect per-HP breakaway test windows automatically for a date range.

    Marks rows where the state is one of the *active* test states
    (MOVINGNEGATIVE, TESTINGPOSITIVE, TESTINGNEGATIVE, MOVINGREFERENCE).
    Adjacent intervals closer than *gap_s* are merged; windows shorter than
    *min_duration_s* are discarded.

    Returns:
        pd.DataFrame with columns:
            date, hp, t_start_utc, t_end_utc, duration_s
    """
    # Only these states mean the HP is actively under test; PASSED / FAILED
    # are excluded so a finished HP does not keep the window open.
    ACTIVE_STATES = {
        int(HardpointTest.MOVINGNEGATIVE),
        int(HardpointTest.TESTINGPOSITIVE),
        int(HardpointTest.TESTINGNEGATIVE),
        int(HardpointTest.MOVINGREFERENCE),
    }

    t0_global = pd.Timestamp(date_start, tz="UTC")
    # Add one full day so date_end is inclusive
    t1_global = pd.Timestamp(date_end, tz="UTC") + pd.Timedelta(days=1)

    if verbose:
        print(
            f"Fetching hardpointTestStatus from {t0_global.date()} "
            f"to {t1_global.date()} ..."
        )

    raw = await efd_client.select_time_series(
        topic_name=TEST_EVENT_TOPIC,
        fields=[f"testState{i}" for i in range(N_HP)],
        start=Time(t0_global.to_pydatetime(), scale="utc"),
        end=Time(t1_global.to_pydatetime(), scale="utc"),
    )

    if raw is None or raw.empty:
        print("  No status data found for this range.")
        return pd.DataFrame()

    raw = ensure_utc_index(raw)
    rows = []

    for hp in range(1, N_HP + 1):
        col = f"testState{hp - 1}"
        if col not in raw.columns:
            continue
        s = raw[col].dropna().astype(int)
        if s.empty:
            continue

        # Boolean mask: True only while this HP is in an active test phase
        active = s.isin(ACTIVE_STATES)
        # diff() on the 0/1 series gives +1 at rising edges, -1 at falling edges
        edges = active.astype(int).diff().fillna(0)
        starts = s.index[edges == 1].tolist()
        ends = s.index[edges == -1].tolist()

        # Handle series that starts or ends mid-activity
        if active.iloc[0]:
            starts = [s.index[0]] + starts
        if active.iloc[-1]:
            ends = ends + [s.index[-1]]

        if not starts or not ends:
            continue

        # Greedily pair each start with the next available end
        pairs, ei = [], 0
        for st in starts:
            while ei < len(ends) and ends[ei] <= st:
                ei += 1
            if ei < len(ends):
                pairs.append((st, ends[ei]))
                ei += 1

        if not pairs:
            continue

        # Merge consecutive windows separated by less than gap_s seconds
        # (brief NOTTESTED gaps between HPs in the same run)
        merged = [pairs[0]]
        for a, b in pairs[1:]:
            prev_a, prev_b = merged[-1]
            if (a - prev_b).total_seconds() < gap_s:
                merged[-1] = (prev_a, b)  # extend previous window
            else:
                merged.append((a, b))

        for a, b in merged:
            dur = (b - a).total_seconds()
            if dur >= min_duration_s:
                rows.append(
                    {
                        "date": a.strftime("%Y-%m-%d"),
                        "hp": hp,
                        "t_start_utc": a,
                        "t_end_utc": b,
                        "duration_s": round(dur, 1),
                    }
                )

    df_windows = (
        pd.DataFrame(rows).sort_values(["t_start_utc", "hp"]).reset_index(drop=True)
    )

    if verbose and not df_windows.empty:
        for day, grp in df_windows.groupby("date"):
            print(f"\n  {day}  →  {len(grp)} test window(s) detected")
            for _, r in grp.iterrows():
                print(
                    f"    HP{r.hp}  {r.t_start_utc.strftime('%H:%M:%S')} – "
                    f"{r.t_end_utc.strftime('%H:%M:%S')} UTC  "
                    f"({r.duration_s:.0f} s)"
                )

    print(f"\nTotal windows found: {len(df_windows)}")
    return df_windows

In [ ]:
# MAIN PLOTTING FUNCTION


async def run_day_plots(
    day_str: str,
    hp_time_windows: dict,
    outdir: str | None = None,
    el_str: str | None = None,
    az_str: str | None = None,
    salindex: str | None = None,
    show_fit_detail: bool = False,
) -> pd.DataFrame:
    """Process and plot stiffness + residual for every HP window on a given day.

    Args:
        day_str         : 'YYYY-MM-DD'
        hp_time_windows : {1: ('05:44:00', '05:54:00'), ...}
        el_str, az_str  : Elevation/azimuth labels (auto-fetched when None).
        salindex        : SAL index label (default 'N/A').
        show_fit_detail : Add extra panel showing OLS fit window detail.

    Returns:
        pd.DataFrame  one row per HP with stiffness values.
    """

    def fmt(v):
        # Format a stiffness float for the on-plot annotation box
        try:
            v = float(v)
            return "NaN" if not np.isfinite(v) else f"{v:.2f}"
        except Exception:
            return "NaN"

    results = []

    hp_keys = [hp for hp in range(1, N_HP + 1) if hp in hp_time_windows]
    if not hp_keys:
        return pd.DataFrame()

    # Compute the bounding window that covers all HPs in this session so we
    # can issue a single status query and a single batch of actuator queries.
    all_t0 = min(
        pd.Timestamp(f"{day_str} {hp_time_windows[hp][0]}", tz="UTC") for hp in hp_keys
    )
    all_t1 = max(
        pd.Timestamp(f"{day_str} {hp_time_windows[hp][1]}", tz="UTC") for hp in hp_keys
    )

    # Fetch status once for the whole session; all actuators concurrently
    status_all, act_all = await asyncio.gather(
        fetch_status(all_t0, all_t1),
        fetch_all_actuators(all_t0, all_t1),
    )

    for hp in hp_keys:
        start_hms, end_hms = hp_time_windows[hp]
        t0 = pd.Timestamp(f"{day_str} {start_hms}", tz="UTC")
        t1 = pd.Timestamp(f"{day_str} {end_hms}", tz="UTC")
        hp_idx = hp - 1

        # Pointing angles: use caller-supplied strings or query the EFD
        if el_str is None or az_str is None:
            el_deg, az_deg = await fetch_elevation_azimuth(t0, t1)
            el = f"{el_deg:.2f}" if np.isfinite(el_deg) else "N/A"
            az = f"{az_deg:.2f}" if np.isfinite(az_deg) else "N/A"
        else:
            el, az = el_str, az_str
            try:
                el_deg, az_deg = float(el_str), float(az_str)
            except ValueError:
                el_deg, az_deg = float("nan"), float("nan")

        salindex = salindex if salindex is not None else DEFAULT_SALINDEX
        print(f"  HP{hp}  El={el} deg  Az={az} deg")

        # Slice the pre-fetched DataFrames to this HP's individual time window
        status_df = (
            status_all[(status_all.index >= t0) & (status_all.index <= t1)]
            if not status_all.empty
            else pd.DataFrame()
        )
        act_df = act_all.get(hp_idx, pd.DataFrame())
        act_df = (
            act_df[(act_df.index >= t0) & (act_df.index <= t1)]
            if not act_df.empty
            else pd.DataFrame()
        )

        if status_df.empty or act_df.empty:
            print(f"  HP{hp}: missing data — skipping")
            continue

        state_col = f"testState{hp_idx}"
        if state_col not in status_df.columns:
            print(f"  HP{hp}: state column missing — skipping")
            continue

        segs = build_state_segments(status_df[state_col], t0, t1)
        # Pick the longest run of each relevant state for fitting
        seg_mn = pick_segment_for_state(segs, int(HardpointTest.MOVINGNEGATIVE))
        seg_tp = pick_segment_for_state(segs, int(HardpointTest.TESTINGPOSITIVE))
        seg_tn = pick_segment_for_state(segs, int(HardpointTest.TESTINGNEGATIVE))

        mn_data = slice_actuator_by_segment(act_df, seg_mn)
        tp_data = slice_actuator_by_segment(act_df, seg_tp)
        tn_data = slice_actuator_by_segment(act_df, seg_tn)

        if mn_data.empty or tp_data.empty or tn_data.empty:
            print(
                f"  HP{hp}: incomplete states  "
                f"mn={len(mn_data)} pos={len(tp_data)} neg={len(tn_data)}"
            )
            continue

        # Stiffness fits
        stiffs, fit_infos, df_plots = {}, {}, {}
        for df_s, label, _ in [
            (mn_data, "Moving Negative", HardpointTest.MOVINGNEGATIVE),
            (tp_data, "Testing Positive", HardpointTest.TESTINGPOSITIVE),
            (tn_data, "Testing Negative", HardpointTest.TESTINGNEGATIVE),
        ]:
            k, df_plot, info = fit_stiffness_centered(df_s)
            stiffs[label] = k
            fit_infos[label] = info
            df_plots[label] = df_plot

        # FIGURE 1: Stiffness
        ncols = 2 if show_fit_detail else 1
        fig, axes = plt.subplots(1, ncols, figsize=(10 * ncols, 8), squeeze=False)
        ax = axes[0, 0]

        ax.axhspan(COMP_LIMITS[0], COMP_LIMITS[1], color="lightgrey", alpha=0.30)
        ax.axhspan(TENS_LIMITS[0], TENS_LIMITS[1], color="yellow", alpha=0.15)
        ax.text(
            -DISPLACEMENT_CROP_RANGE * 0.85,
            np.mean(COMP_LIMITS),
            "Compression",
            ha="left",
            va="center",
            fontsize=9,
            fontweight="bold",
        )
        ax.text(
            -DISPLACEMENT_CROP_RANGE * 0.85,
            np.mean(TENS_LIMITS),
            "Tension",
            ha="left",
            va="center",
            fontsize=9,
            fontweight="bold",
        )

        for label, st_enum in [
            ("Moving Negative", HardpointTest.MOVINGNEGATIVE),
            ("Testing Positive", HardpointTest.TESTINGPOSITIVE),
            ("Testing Negative", HardpointTest.TESTINGNEGATIVE),
        ]:
            k = stiffs[label]
            df_plot = df_plots[label]
            c = STATUS_COLORS[st_enum]
            # Thin markers: show one every ~120 points to keep the plot readable
            step = max(1, len(df_plot) // 120)
            ax.plot(
                df_plot["displacement"],
                df_plot["force"],
                "-",
                color=c,
                alpha=1.0,
                linewidth=2,
                marker="o",
                markersize=4,
                markevery=step,
                markeredgewidth=0,
                label=label,
            )
            if np.isfinite(k):
                # Overlay the OLS fit line over the full plot range
                fit_x = df_plot["displacement"].to_numpy(float)
                ax.plot(fit_x, k * fit_x, "-", color=c, linewidth=2, label="_nolegend_")

        x_spec = np.linspace(-DISPLACEMENT_CROP_RANGE, DISPLACEMENT_CROP_RANGE, 200)
        ax.plot(
            x_spec,
            x_spec * SPEC_STIFFNESS,
            ":",
            color="black",
            linewidth=2,
            label=f"Spec ({SPEC_STIFFNESS} N/µm)",
        )
        ax.axvline(0, color="black", linewidth=1, alpha=0.5)
        ax.axhline(0, color="black", linewidth=1, alpha=0.5)
        ax.set_xlim(-DISPLACEMENT_CROP_RANGE - 50, DISPLACEMENT_CROP_RANGE + 50)
        ax.set_ylim(-6000, 6000)
        ax.set_xlabel("Displacement [µm]", fontsize=11)
        ax.set_ylabel("Force [N]", fontsize=11)
        ax.grid(True, alpha=0.25, linestyle="--")
        ax.tick_params(axis="both", labelsize=10)
        ax.set_title(
            f"Individual Hardpoint Breakaway Test at El:{el} deg, Az:{az} deg\n"
            f"HP{hp} Stiffness | {day_str} "
            f"{t0.strftime('%H:%M:%S')}–{t1.strftime('%H:%M:%S')} UTC",
            fontsize=11,
        )
        leg = ax.legend(loc="upper left", fontsize=9, frameon=True)
        leg.get_frame().set_alpha(0.9)

        stiff_text = (
            "Stiffness N/µm\n"
            f"Moving Negative:  {fmt(stiffs.get('Moving Negative'))}\n"
            f"Testing Positive: {fmt(stiffs.get('Testing Positive'))}\n"
            f"Testing Negative: {fmt(stiffs.get('Testing Negative'))}"
        )
        ax.text(
            0.98,
            0.02,
            stiff_text,
            transform=ax.transAxes,
            ha="right",
            va="bottom",
            fontsize=9,
            bbox=dict(
                boxstyle="round,pad=0.4", facecolor="white", alpha=0.9, edgecolor="none"
            ),
        )

        # Optional: fit detail panel
        if show_fit_detail:
            ax2 = axes[0, 1]
            for label, st_enum in [
                ("Testing Positive", HardpointTest.TESTINGPOSITIVE),
                ("Testing Negative", HardpointTest.TESTINGNEGATIVE),
            ]:
                color = STATUS_COLORS[st_enum]
                k_val = stiffs.get(label, np.nan)
                df_plot = df_plots.get(label, pd.DataFrame())
                if df_plot.empty:
                    continue
                # Restrict scatter to the actual fit window for clarity
                df_win = df_plot[
                    df_plot["displacement"].abs() <= DISPLACEMENT_CROP_RANGE_FOR_FIT
                ]
                ax2.scatter(
                    df_win["displacement"],
                    df_win["force"],
                    s=14,
                    color=color,
                    alpha=0.6,
                    label=f"{label}  ({len(df_win)} pts)",
                )
                if np.isfinite(k_val):
                    xl = np.linspace(
                        df_win["displacement"].min(), df_win["displacement"].max(), 100
                    )
                    ax2.plot(
                        xl,
                        k_val * xl,
                        color=color,
                        lw=2,
                        label=f"{label}  k={k_val:.3f} N/µm",
                    )
            ax2.axhline(0, color="grey", lw=0.8)
            ax2.axvline(0, color="grey", lw=0.8)
            ax2.axvline(
                DISPLACEMENT_CROP_RANGE_FOR_FIT,
                color="steelblue",
                lw=1,
                ls=":",
                alpha=0.6,
                label=f"±{DISPLACEMENT_CROP_RANGE_FOR_FIT} µm fit window",
            )
            ax2.axvline(
                -DISPLACEMENT_CROP_RANGE_FOR_FIT,
                color="steelblue",
                lw=1,
                ls=":",
                alpha=0.6,
            )
            ax2.set_xlabel("Displacement (µm)", fontsize=10)
            ax2.set_ylabel("Force (N)", fontsize=10)
            ax2.set_title(
                f"Fit window detail — HP{hp}  ±{DISPLACEMENT_CROP_RANGE_FOR_FIT:.0f} µm",
                fontsize=10,
            )
            ax2.grid(True, alpha=0.2, linestyle="--")
            ax2.legend(fontsize=7.5, frameon=False)

        plt.tight_layout()
        plt.show()
        save_fig(
            fig,
            outdir,
            f"stiffness_HP{hp}_{day_str}_El{el}.png",
        )
        plt.close(fig)

        # FIGURE 2: ERF residual
        # Centre both directions independently before fitting the erf model
        tp_c = center_df_force_disp(tp_data[["force", "displacement"]])
        tn_c = center_df_force_disp(tn_data[["force", "displacement"]])

        fig2, (ax_fd, ax_res) = plt.subplots(
            1, 2, figsize=(10, 4), gridspec_kw={"width_ratios": [1, 1]}
        )
        title2 = (
            f"Individual Hardpoint Breakaway Test at El:{el} deg, Az:{az} deg\n"
            f"HP{hp} | {day_str} "
            f"{t0.strftime('%H:%M:%S')}–{t1.strftime('%H:%M:%S')} UTC"
        )
        plot_error_residual(ax_fd, ax_res, tp_c, tn_c, title=title2, max_markers=90)
        ax_fd.set_xlim(-DISPLACEMENT_CROP_RANGE - 50, DISPLACEMENT_CROP_RANGE + 50)
        ax_fd.set_ylim(-6000, 6000)
        ax_res.set_xlim(-DISPLACEMENT_CROP_RANGE - 50, DISPLACEMENT_CROP_RANGE + 50)
        ax_res.set_ylim(-1000, 1000)
        plt.tight_layout()
        plt.show()
        save_fig(
            fig2,
            outdir,
            f"residual_HP{hp}_{day_str}_El{el}.png",
        )
        plt.close(fig2)

        # Results row
        results.append(
            {
                "Date": day_str,
                "SALIndex": salindex,
                "Hardpoint": f"HP{hp}",
                "Start_UTC": t0.strftime("%Y-%m-%d %H:%M:%S"),
                "End_UTC": t1.strftime("%Y-%m-%d %H:%M:%S"),
                "Elevation_deg": el_deg,
                "Azimuth_deg": az_deg,
                "Stiffness_MovingNegative": stiffs.get("Moving Negative", np.nan),
                "Stiffness_TestingPositive": stiffs.get("Testing Positive", np.nan),
                "Stiffness_TestingNegative": stiffs.get("Testing Negative", np.nan),
            }
        )

    return pd.DataFrame(results) if results else pd.DataFrame()

In [ ]:
# FORCE VS TIME — per HP per session


async def plot_force_vs_time(
    day_str: str,
    hp_time_windows: dict,
    outdir: str | None = None,
    salindex: str | None = None,
) -> pd.DataFrame:
    """Plot measured force vs time for each HP in a session.

    Returns:
        pd.DataFrame  one row per (HP, state segment) with force stats.
    """
    PLOT_STATES = {
        int(HardpointTest.MOVINGNEGATIVE),
        int(HardpointTest.TESTINGPOSITIVE),
        int(HardpointTest.TESTINGNEGATIVE),
        int(HardpointTest.MOVINGREFERENCE),
    }
    # Reverse mapping used to recover the enum from its integer value
    INT_TO_ENUM = {int(e): e for e in HardpointTest}
    salindex = salindex if salindex is not None else DEFAULT_SALINDEX
    all_rows = []

    hp_keys = [hp for hp in range(1, N_HP + 1) if hp in hp_time_windows]
    if not hp_keys:
        return pd.DataFrame()

    # Single bounding window for batch EFD fetch (same pattern as run_day_plots)
    all_t0 = min(
        pd.Timestamp(f"{day_str} {hp_time_windows[hp][0]}", tz="UTC") for hp in hp_keys
    )
    all_t1 = max(
        pd.Timestamp(f"{day_str} {hp_time_windows[hp][1]}", tz="UTC") for hp in hp_keys
    )

    status_all, act_all = await asyncio.gather(
        fetch_status(all_t0, all_t1),
        fetch_all_actuators(all_t0, all_t1),
    )

    for hp in hp_keys:
        start_hms, end_hms = hp_time_windows[hp]
        t0 = pd.Timestamp(f"{day_str} {start_hms}", tz="UTC")
        t1 = pd.Timestamp(f"{day_str} {end_hms}", tz="UTC")
        hp_idx = hp - 1

        el_deg, az_deg = await fetch_elevation_azimuth(t0, t1)
        el_str = f"{el_deg:.2f}" if np.isfinite(el_deg) else "N/A"
        az_str = f"{az_deg:.2f}" if np.isfinite(az_deg) else "N/A"

        # Slice pre-fetched data to this HP's individual time window
        act_df = act_all.get(hp_idx, pd.DataFrame())
        act_df = (
            act_df[(act_df.index >= t0) & (act_df.index <= t1)]
            if not act_df.empty
            else pd.DataFrame()
        )
        status_df = (
            status_all[(status_all.index >= t0) & (status_all.index <= t1)]
            if not status_all.empty
            else pd.DataFrame()
        )

        if act_df.empty or status_df.empty:
            print(f"  HP{hp}: no data — skipping")
            continue

        state_col = f"testState{hp_idx}"
        if state_col not in status_df.columns:
            continue

        segs = build_state_segments(status_df[state_col], t0, t1)
        # Keep only segments whose integer state is in PLOT_STATES and that
        # are longer than 1 s (filters out sub-sample transition artefacts)
        filtered_segments = [
            (a, b, INT_TO_ENUM[st])
            for (a, b, st) in segs
            if st in PLOT_STATES and (b - a).total_seconds() > 1
        ]
        if not filtered_segments:
            print(f"  HP{hp}: no active segments found — skipping")
            continue

        x_min = min(a for a, b, _ in filtered_segments)
        x_max = max(b for a, b, _ in filtered_segments)
        margin = pd.Timedelta("30s")

        fig, ax = plt.subplots(figsize=(18, 7))
        ax.axhspan(
            COMP_LIMITS[0],
            COMP_LIMITS[1],
            color="lightgrey",
            alpha=0.5,
            label="Compression Range",
        )
        ax.axhspan(
            TENS_LIMITS[0],
            TENS_LIMITS[1],
            color="yellow",
            alpha=0.15,
            label="Tension Range",
        )

        labels_added = set()

        for seg_start, seg_end, state_enum in filtered_segments:
            mask = (act_df.index >= seg_start) & (act_df.index <= seg_end)
            data = act_df.loc[mask, "force"]
            if data.empty:
                continue
            # Suppress duplicate legend entries for the same state
            lbl = state_enum.name if state_enum.name not in labels_added else None
            ax.plot(
                data.index,
                data.values,
                color=STATUS_COLORS[state_enum],
                label=lbl,
                linewidth=2.5,
            )
            # Vertical dashed line marks each state transition boundary
            ax.axvline(seg_start, color="black", linestyle=":", alpha=0.6, linewidth=1)
            labels_added.add(state_enum.name)

            all_rows.append(
                {
                    "Date": day_str,
                  # "SALIndex": salindex,
                    "Hardpoint": f"HP{hp}",
                    "Elevation_deg": el_deg,
                    "Azimuth_deg": az_deg,
                    "State": state_enum.name,
                    "Segment_Start_UTC": seg_start.strftime("%Y-%m-%d %H:%M:%S"),
                    "Segment_End_UTC": seg_end.strftime("%Y-%m-%d %H:%M:%S"),
                    "Duration_s": round((seg_end - seg_start).total_seconds(), 1),
                    "Max_Force_N": round(float(data.max()), 2),
                    "Min_Force_N": round(float(data.min()), 2),
                }
            )

        x_right = x_max + margin
        # Annotate acceptance-band edge values on the right side of the plot
        for val in [COMP_LIMITS[0], COMP_LIMITS[1], TENS_LIMITS[0], TENS_LIMITS[1]]:
            ax.text(
                x_right + pd.Timedelta("2s"), val, f"{val} N", fontsize=9, va="center"
            )

        ax.text(
            0.02,
            np.mean(COMP_LIMITS),
            "Compression",
            transform=ax.get_yaxis_transform(),
            ha="left",
            va="center",
            fontweight="bold",
            fontsize=10,
        )
        ax.text(
            0.02,
            np.mean(TENS_LIMITS),
            "Tension",
            transform=ax.get_yaxis_transform(),
            ha="left",
            va="center",
            fontweight="bold",
            fontsize=10,
        )

        ax.set_xlim(x_min - margin, x_right)
        ax.set_ylim(-4500, 4000)
        ax.set_title(
            f"Individual Hardpoint Breakaway Test at El:{el_str} deg, Az:{az_str} deg\n"
            f"HP{hp} | {day_str}  "
            f"{x_min.strftime('%H:%M:%S')} – {x_max.strftime('%H:%M:%S')} UTC",
            fontsize=12,
        )
        ax.set_ylabel(f"HP{hp} Measured Force [N]", fontsize=11)
        ax.set_xlabel("Time [UTC]", fontsize=11)
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M:%S"))
        ax.grid(True, alpha=0.3)
        if labels_added:
            ax.legend(loc="upper right", fontsize=10, frameon=True)

        plt.show()
        save_fig(
            fig,
            outdir,
            f"force_time_HP{hp}_{day_str}_El{el_str}.png",
        )
        plt.close(fig)

    return pd.DataFrame(all_rows) if all_rows else pd.DataFrame()

In [ ]:
# SESSION SPLITTER + DATE-RANGE RUNNER


def split_into_sessions(grp: pd.DataFrame, gap_s: float = SESSION_GAP_S) -> list:
    """Split a per-day group of HP windows into distinct test sessions.

    A new session is declared when the gap between consecutive window start
    times exceeds *gap_s* seconds.
    """
    grp = grp.sort_values("t_start_utc").reset_index(drop=True)
    sessions = []
    session_rows = [grp.iloc[0]]

    for i in range(1, len(grp)):
        prev_end = session_rows[-1]["t_end_utc"]
        this_start = grp.iloc[i]["t_start_utc"]
        if (this_start - prev_end).total_seconds() > gap_s:
            # Gap exceeds threshold → close current session and start a new one
            sessions.append(pd.DataFrame(session_rows))
            session_rows = []
        session_rows.append(grp.iloc[i])

    if session_rows:
        sessions.append(pd.DataFrame(session_rows))
    return sessions


async def run_date_range(
    date_start: str,
    date_end: str,
    outdir: str | None = None,
    show_fit_detail: bool = False,
    save_df: bool = False,
    save_df_path: str | None = None,
    save_df_fmt: str = "csv",
) -> pd.DataFrame:
    """Detect windows automatically and run plots for every session in the range.

    Args:
        date_start    : 'YYYY-MM-DD'  first day (inclusive)
        date_end      : 'YYYY-MM-DD'  last  day (inclusive)
        outdir        : root directory for PNG output; None disables saving.
        show_fit_detail : pass-through to run_day_plots.
        save_df       : if True, persist the results DataFrame to disk.
        save_df_path  : output path for the DataFrame.
                        Defaults to '<outdir>/results_<date_start>_<date_end>.<fmt>'
                        when *outdir* is set, otherwise './results_<dates>.<fmt>'.
        save_df_fmt   : 'csv' (default) or 'parquet'.

    Returns:
        pd.DataFrame  consolidated results for all sessions.
    """
    df_windows = await detect_test_windows(date_start, date_end)

    if df_windows.empty:
        print("No test windows found — nothing to plot.")
        return pd.DataFrame()

    all_results = []

    # Group by calendar date; within each date split into independent sessions
    for day_str, day_grp in df_windows.groupby("date"):
        sessions = split_into_sessions(day_grp)

        print(f"\n{'='*60}")
        print(
            f"  {day_str}  —  {len(sessions)} session(s), "
            f"{len(day_grp)} HP window(s) total"
        )
        print(f"{'='*60}")

        for s_idx, session_grp in enumerate(sessions, start=1):
            t_sess_start = session_grp["t_start_utc"].min().strftime("%H:%M:%S")
            t_sess_end = session_grp["t_end_utc"].max().strftime("%H:%M:%S")
            print(
                f"\n  Session {s_idx}/{len(sessions)}  "
                f"{t_sess_start} – {t_sess_end} UTC  "
                f"({len(session_grp)} HPs)"
            )

            # Build per-session hp_time_windows dict; no key collisions between
            # sessions because each is processed in its own loop iteration.
            hp_time_windows = {
                int(r.hp): (
                    r.t_start_utc.strftime("%H:%M:%S"),
                    r.t_end_utc.strftime("%H:%M:%S"),
                )
                for _, r in session_grp.iterrows()
            }

            session_outdir = (
                os.path.join(outdir, day_str, f"session_{s_idx:02d}")
                if outdir
                else None
            )

            df_sess = await run_day_plots(
                day_str=day_str,
                hp_time_windows=hp_time_windows,
                outdir=session_outdir,
                show_fit_detail=show_fit_detail,
            )
            if df_sess is not None and not df_sess.empty:
                # Tag each row with its session index for later traceability
                df_sess.insert(1, "Session", s_idx)
                all_results.append(df_sess)

            await plot_force_vs_time(
                day_str=day_str,
                hp_time_windows=hp_time_windows,
                outdir=session_outdir,
            )

    df_all = (
        pd.concat(all_results, ignore_index=True) if all_results else pd.DataFrame()
    )

    # Optional: save results DataFrame
    if save_df and not df_all.empty:
        if save_df_path is None:
            # Auto-generate path from outdir and date range
            base = outdir if outdir else "."
            tag = f"{date_start}_{date_end}"
            ext = ".parquet" if save_df_fmt == "parquet" else ".csv"
            save_df_path = os.path.join(base, f"results_{tag}{ext}")
        save_dataframe(df_all, save_df_path, fmt=save_df_fmt)

    return df_all

## Usage Modes

This script offers three distinct execution modes depending on the desired output:

1. **Date Range Scan** (*Metadata only*)
    - Description: Scans a specific range of dates to identify test days without generating visualizations.
    - Output: Summary of available test dates in the console or log.
2. **Full Analysis** (*Plots & Data*)
    - Description: Performs a complete scan of the selected date range.
    - Output: Generates a CSV file with extracted features and saves all associated plots (Force vs. Time, Stiffness, and
      Residuals) to the output directory.
3. **Data Export** (*Format conversion*)
    - Description: Processes the internal DataFrame and exports it to a structured file.
    - Output: Saves the data in CSV or Parquet format for use in external tools or future analysis.

In [ ]:
# 1. DATE-RANGE

df_windows = await detect_test_windows("2024-03-01", "2024-03-10")
display(df_windows)

In [ ]:
# 2. FULL ANALYSIS + PLOTS FOR THE DATE RANGE

df_all = await run_date_range(
    date_start="2023-05-26",
    date_end="2023-06-20",
    outdir="plots/may_june_2023",
    show_fit_detail=False,
    save_df=False,
    save_df_fmt="csv",
    # save_df_path = "my_path/results.csv",  # personalized path (optional)
)

display(df_all)

In [ ]:
# 3. DATAFRAME EXPORT

save_dataframe(
    df_all, path="results/stiffness_may2023.csv", fmt="csv"
)  # fmt can be 'csv' or 'parquet'